### 1- Load And Explore the Dataset

In [ ]:
# ربط Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

# حددي المسار حسب مكان الملف داخل Google Drive
file_path = '/content/drive/MyDrive/Financial_fraud_detection_project/Fraud_data/PS_20174392719_1491204439457_log.csv'

# قراءة البيانات
df = pd.read_csv(file_path)

# عرض أول 5 صفوف
df.head()


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [ ]:
# # حساب التكرار لكل قيمة في العمود
# counts = df['isFraud'].value_counts()

# # حساب النسب المئوية
# percentages = df['isFraud'].value_counts(normalize=True) * 100

# # طباعة النتائج
# for value in counts.index:
#     print(f"القيمة {value}:")
#     print(f"  عدد الصفوف: {counts[value]}")
#     print(f"  النسبة المئوية: {percentages[value]:.2f}%")


In [ ]:
# # نفترض الداتا فريم الأصلي اسمه df
# target_size = 40387

# # target_size = 60387
# fraud_ratio = 0.033  # 3.3%

# num_fraud = int(target_size * fraud_ratio)          # عدد عينات فراود 3.3%
# num_non_fraud = target_size - num_fraud             # باقي العينات بدون فراود

# # أخذ عينات عشوائية من كل فئة
# fraud_samples = df[df['isFraud'] == 1].sample(n=num_fraud, random_state=42)
# non_fraud_samples = df[df['isFraud'] == 0].sample(n=num_non_fraud, random_state=42)

# # دمج وخلط العينات عشوائياً
# sampled_df = pd.concat([fraud_samples, non_fraud_samples]).sample(frac=1, random_state=42).reset_index(drop=True)


In [ ]:
# sampled_df.to_excel('sample_for_dashboard.xlsx', index=False)

In [ ]:
# حجم البيانات
print("عدد الصفوف:", df.shape[0])
print("عدد الأعمدة:", df.shape[1])

# أسماء الأعمدة وأنواعها
print("\nأنواع البيانات:")
print(df.dtypes)

# هل فيه بيانات ناقصة؟
print("\nالقيم المفقودة:")
print(df.isnull().sum())


عدد الصفوف: 6362620
عدد الأعمدة: 11

أنواع البيانات:
step                int64
type               object
amount            float64
nameOrig           object
oldbalanceOrg     float64
newbalanceOrig    float64
nameDest           object
oldbalanceDest    float64
newbalanceDest    float64
isFraud             int64
isFlaggedFraud      int64
dtype: object

القيم المفقودة:
step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64


In [ ]:
# أنواع العمليات المالية
print("أنواع المعاملات المالية (type):", df['type'].unique())

# عدد العمليات الاحتيالية
print("\nعدد العمليات الاحتيالية:", df['isFraud'].sum())
print("عدد العمليات التي تم الاشتباه بها:", df['isFlaggedFraud'].sum())


أنواع المعاملات المالية (type): ['PAYMENT' 'TRANSFER' 'CASH_OUT' 'DEBIT' 'CASH_IN']

عدد العمليات الاحتيالية: 8213
عدد العمليات التي تم الاشتباه بها: 16


### 2- Features Engineering and Split the data

In [ ]:
df_encoded = pd.get_dummies(df, columns=['type'], drop_first=True)

# المتغيرات المستقلة (Features)
features = [
    'step', 'amount', 'oldbalanceOrg', 'newbalanceOrig',
    'oldbalanceDest', 'newbalanceDest',
    'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER'
]

# المتغير الهدف (Label)
target = 'isFraud'

X = df_encoded[features]
y = df_encoded[target]


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


### 3.1- Random forest model training

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, random_state=42)

model.fit(X_train, y_train)


RandomForestClassifier(random_state=42)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270904
           1       0.98      0.80      0.88      1620

    accuracy                           1.00   1272524
   macro avg       0.99      0.90      0.94   1272524
weighted avg       1.00      1.00      1.00   1272524

Confusion Matrix:
[[1270874      30]
 [    316    1304]]


### 3.2- Logistic regression model training

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

print("=== Logistic Regression ===")
print(classification_report(y_test, y_pred_lr))


=== Logistic Regression ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270904
           1       0.91      0.49      0.63      1620

    accuracy                           1.00   1272524
   macro avg       0.96      0.74      0.82   1272524
weighted avg       1.00      1.00      1.00   1272524



### 3.3- XG_Boost model training

In [ ]:
!pip install xgboost

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix

xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)

print("=== XGBoost ===")
print(classification_report(y_test, y_pred_xgb))


/usr/local/lib/python3.11/dist-packages/xgboost/training.py:183: UserWarning: [17:41:12] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


=== XGBoost ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270904
           1       0.49      0.42      0.45      1620

    accuracy                           1.00   1272524
   macro avg       0.74      0.71      0.73   1272524
weighted avg       1.00      1.00      1.00   1272524



### 3.3.2- XG_Boost Enhancement

In [ ]:
# توقع احتمالات بدلاً من نتائج نهائية
y_probs = xgb.predict_proba(X_test)[:, 1]

# نحط حد جديد للتصنيف (مثلاً 0.3 بدل 0.5)
import numpy as np
y_pred_thresh = np.where(y_probs > 0.3, 1, 0)

# نشوف الأداء
from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(y_test, y_pred_thresh))
print(confusion_matrix(y_test, y_pred_thresh))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270904
           1       0.48      0.46      0.47      1620

    accuracy                           1.00   1272524
   macro avg       0.74      0.73      0.73   1272524
weighted avg       1.00      1.00      1.00   1272524

[[1270111     793]
 [    880     740]]


In [ ]:
xgb = XGBClassifier(
    scale_pos_weight=100,  # يساعد في تعويض التوازن
    max_depth=5,
    learning_rate=0.1,
    n_estimators=100,
    eval_metric='logloss',
    use_label_encoder=False
)

xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)

print("=== XGBoost ===")
print(classification_report(y_test, y_pred_xgb))



/usr/local/lib/python3.11/dist-packages/xgboost/training.py:183: UserWarning: [17:42:22] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


=== XGBoost ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270904
           1       0.31      0.99      0.47      1620

    accuracy                           1.00   1272524
   macro avg       0.65      0.99      0.73   1272524
weighted avg       1.00      1.00      1.00   1272524



### 3.3.3- Export the GL_Boost Model

### 3.4- LGBM_Classifier model training

In [ ]:
!pip install lightgbm

In [ ]:
from lightgbm import LGBMClassifier

lgb = LGBMClassifier()
lgb.fit(X_train, y_train)
y_pred_lgb = lgb.predict(X_test)

print("=== LightGBM ===")
print(classification_report(y_test, y_pred_lgb))


[LightGBM] [Info] Number of positive: 6593, number of negative: 5083503
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.864703 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1538
[LightGBM] [Info] Number of data points in the train set: 5090096, number of used features: 10
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.001295 -> initscore=-6.647747
[LightGBM] [Info] Start training from score -6.647747
=== LightGBM ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270904
           1       0.54      0.62      0.58      1620

    accuracy                           1.00   1272524
   macro avg       0.77      0.81      0.79   1272524
weighted avg       1.00      1.00      1.00   1272524



In [ ]:
import joblib
joblib.dump(xgb, 'xgb_model.pkl')


['xgb_model.pkl']